## Graph Analysis - DDoS Friday Dataset

In [9]:
from pathlib import Path
from data_loader import load_cicids_folder, standardize_flow_columns, build_node_majority_labels
from graph_builder import build_weighted_graph
import pandas as pd

# Load directly from CSV (bypass pickle issues)
flows = standardize_flow_columns(load_cicids_folder(Path("data")))
labels = build_node_majority_labels(flows)
graph = build_weighted_graph(flows)

print("=" * 70)
print("GRAPH STRUCTURE")
print("=" * 70)
print(f"Flows: {len(flows)} rows")
print(f"Labeled Nodes: {len(labels)} nodes")
print(f"Graph Nodes: {graph.number_of_nodes()}")
print(f"Graph Edges: {graph.number_of_edges()}")
print(f"Unlabeled nodes: {graph.number_of_nodes() - len(labels)}")

GRAPH STRUCTURE
Flows: 225745 rows
Labeled Nodes: 2067 nodes
Graph Nodes: 2567
Graph Edges: 6963
Unlabeled nodes: 500


In [12]:
# Simple attacker/victim check from DDoS-only rows
print("\n" + "=" * 70)
print("ATTACKER / VICTIM CHECK (DDoS ONLY)")
print("=" * 70)

ddos = flows[flows['label'].str.upper() == 'DDOS'].copy()
print(f"DDoS rows: {len(ddos)}")

print("\nTop attacker candidates (src in DDoS):")
print(ddos['src'].value_counts().head(10))

print("\nTop victim candidates (dst in DDoS):")
print(ddos['dst'].value_counts().head(10))

print("\nTop attack paths (src -> dst) in DDoS:")
path_counts = (
    ddos.groupby(['src', 'dst']).size()
    .sort_values(ascending=False)
    .head(10)
)
print(path_counts)

print("\nKnown external attacker IPs (presence in DDoS rows):")
known_attackers = ['205.174.165.73', '205.174.165.69', '205.174.165.70', '205.174.165.71']
for ip in known_attackers:
    src_hits = int((ddos['src'] == ip).sum())
    dst_hits = int((ddos['dst'] == ip).sum())
    print(f"{ip}: as src={src_hits}, as dst={dst_hits}")

print("\nInterpretation tip: if external attacker IPs are zero here, this CSV is likely NAT/internal-viewed.")


ATTACKER / VICTIM CHECK (DDoS ONLY)
DDoS rows: 128027

Top attacker candidates (src in DDoS):
src
172.16.0.1       128024
192.168.10.50         3
Name: count, dtype: int64

Top victim candidates (dst in DDoS):
dst
192.168.10.50    128024
172.16.0.1            3
Name: count, dtype: int64

Top attack paths (src -> dst) in DDoS:
src            dst          
172.16.0.1     192.168.10.50    128024
192.168.10.50  172.16.0.1            3
dtype: int64

Known external attacker IPs (presence in DDoS rows):
205.174.165.73: as src=0, as dst=0
205.174.165.69: as src=0, as dst=0
205.174.165.70: as src=0, as dst=0
205.174.165.71: as src=0, as dst=0

Interpretation tip: if external attacker IPs are zero here, this CSV is likely NAT/internal-viewed.


In [13]:
# Verify identity layer: check IP presence in src, dst, and Flow ID
print("\n" + "=" * 70)
print("IP PRESENCE CHECK (src / dst / Flow ID)")
print("=" * 70)

ips_to_check = [
    '205.174.165.68',  # victim public
    '205.174.165.80',  # firewall public
    '205.174.165.73',
    '205.174.165.69',
    '205.174.165.70',
    '205.174.165.71',
    '172.16.0.1',      # internal translated source seen in DDoS
    '192.168.10.50',   # local victim
]

# Read raw CSV once to inspect Flow ID text directly
raw_full = pd.read_csv(Path("data/DDoS-Friday-WorkingHours-Afternoon.pcap_ISCX.csv"), low_memory=False)
flow_id_col = 'Flow ID' if 'Flow ID' in raw_full.columns else [c for c in raw_full.columns if c.strip().lower() == 'flow id'][0]
flow_id_series = raw_full[flow_id_col].astype(str)

for ip in ips_to_check:
    src_count = int((flows['src'] == ip).sum())
    dst_count = int((flows['dst'] == ip).sum())
    flowid_count = int(flow_id_series.str.contains(ip, regex=False).sum())
    print(f"{ip:15} | src={src_count:7} | dst={dst_count:7} | FlowID_contains={flowid_count:7}")

print("\nTip: If an IP appears in Flow ID but not in src/dst, translation occurred before exported endpoint fields.")


IP PRESENCE CHECK (src / dst / Flow ID)
205.174.165.68  | src=      0 | dst=      2 | FlowID_contains=      2
205.174.165.80  | src=      0 | dst=      0 | FlowID_contains=      0
205.174.165.73  | src=      0 | dst=    350 | FlowID_contains=    350
205.174.165.69  | src=      0 | dst=      0 | FlowID_contains=      0
205.174.165.70  | src=      0 | dst=      0 | FlowID_contains=      0
205.174.165.71  | src=      0 | dst=      0 | FlowID_contains=      0
172.16.0.1      | src= 128181 | dst=  31343 | FlowID_contains= 159524
192.168.10.50   | src=  32896 | dst= 128834 | FlowID_contains= 161730

Tip: If an IP appears in Flow ID but not in src/dst, translation occurred before exported endpoint fields.
